# Pré-processamento — análise do subject S1

1. Aplicação de filtros temporais (band pass, low/high pass) e decidir se os dados são usáveis ou não;
2. Comparar antes e depois da aplicação dos filtros;
3. Verificar o notch filter (50/60 Hz).

Só o subject S1 é analisado

In [ ]:
import numpy as np
import scipy.io as sio
from scipy import signal

DATA_DIR = 'dataset'
FS = 250                     # Hz
ONSET = 125                  # amostra do início do estímulo (0,5 s × 250 Hz)
OCC = ['PZ', 'PO3', 'PO5', 'PO4', 'PO6', 'POZ', 'O1', 'OZ', 'O2']   # 9 canais usados no paper do BETA

In [ ]:
def unwrap(v):
    a = np.asarray(v)
    if a.dtype == object:
        a = np.array([x.item() if isinstance(x, np.ndarray) and x.size == 1 else x for x in a.ravel()])
    return a.ravel()


def load_subject(path):
    m = sio.loadmat(path)['data'][0, 0]
    suppl = {k: unwrap(m['suppl_info'][0, 0][k]) for k in m['suppl_info'].dtype.names}
    return m['EEG'], suppl

## 1. Carregamento e visão geral

In [3]:
eeg, info = load_subject(f'{DATA_DIR}/S1.mat')
names = list(info['chan'].astype(str).reshape(64, 4)[:, 3])
freqs = info['freqs']
occ = [names.index(n) for n in OCC]

n_ch, T, n_blocks, n_cond = eeg.shape
print(f"shape: {eeg.shape} -> {n_ch} canais, {T} amostras ({T / FS:.0f} s), {n_blocks} blocos, {n_cond} condições")
print(f"janela de estímulo: amostras {ONSET} a {ONSET + 2 * FS} (2 s)")
print(f"narrow_snr informado no arquivo: {info['narrow_snr'][0]:.2f} dB")

shape: (64, 750, 4, 40) -> 64 canais, 750 amostras (3 s), 4 blocos, 40 condições
janela de estímulo: amostras 125 a 625 (2 s)
narrow_snr informado no arquivo: 4.67 dB


## 2. Dados são usáveis?

Checagens de valores inválidos, canais mortos/ruidosos e ensaios com amplitude exagerada.

In [4]:
print("NaN:", np.isnan(eeg).sum(), "| Inf:", np.isinf(eeg).sum())

std = eeg.std(axis=(1, 2, 3))          # desvio-padrão de cada canal (µV)
med = np.median(std)
print(f"desvio-padrão mediano dos canais: {med:.2f} µV")
for i in range(n_ch):
    if std[i] < med / 3 or std[i] > med * 3:
        print(f"  canal suspeito: {names[i]} (std = {std[i]:.2f} µV)")
print("std dos canais occipitais:", {n: round(float(std[i]), 2) for n, i in zip(OCC, occ)})

NaN: 0 | Inf: 0
desvio-padrão mediano dos canais: 5.96 µV
  canal suspeito: C1 (std = 1.55 µV)
  canal suspeito: CZ (std = 0.43 µV)
  canal suspeito: C2 (std = 1.95 µV)
  canal suspeito: M1 (std = 23.96 µV)
  canal suspeito: CPZ (std = 1.65 µV)
  canal suspeito: M2 (std = 0.51 µV)
std dos canais occipitais: {'PZ': 3.8, 'PO3': 6.41, 'PO5': 6.44, 'PO4': 6.92, 'PO6': 9.52, 'POZ': 6.6, 'O1': 9.92, 'OZ': 10.16, 'O2': 12.21}


In [5]:
LIMITE = 100   # µV
pico = np.abs(eeg).max(axis=1)              # (canal, bloco, condição)
ruins_occ = (pico[occ] > LIMITE).any(axis=0)
ruins_todos = (pico > LIMITE).any(axis=0)
print(f"ensaios com |amplitude| > {LIMITE} µV nos canais occipitais: {ruins_occ.sum()} de 160")
print(f"ensaios com |amplitude| > {LIMITE} µV em qualquer canal:     {ruins_todos.sum()} de 160")

ensaios com |amplitude| > 100 µV nos canais occipitais: 0 de 160
ensaios com |amplitude| > 100 µV em qualquer canal:     48 de 160


## 3. Filtros temporais

Todos Butterworth de 4ª ordem aplicados com `filtfilt` (fase zero), no ensaio inteiro:

- **high-pass 6 Hz**: remove o que fica abaixo do menor estímulo (8 Hz);
- **low-pass 90 Hz**: o paper diz que os harmônicos úteis vão até ~90 Hz;
- **band-pass 6–90 Hz**: os dois juntos;
- **notch 50 Hz** e **notch 60 Hz**: para testar a rede elétrica.

In [ ]:
def highpass(x, fc):
    b, a = signal.butter(4, fc, btype='highpass', fs=FS)
    return signal.filtfilt(b, a, x, axis=1)


def lowpass(x, fc):
    b, a = signal.butter(4, fc, btype='lowpass', fs=FS)
    return signal.filtfilt(b, a, x, axis=1)


def bandpass(x, f1, f2):
    b, a = signal.butter(4, [f1, f2], btype='bandpass', fs=FS)
    return signal.filtfilt(b, a, x, axis=1)


def notch(x, f0):
    b, a = signal.iirnotch(f0, Q=30, fs=FS)
    return signal.filtfilt(b, a, x, axis=1)


import time
inicio = time.time()

x = eeg[occ]                     # apenas canais occipitais
versoes = {
    'original': x,
    'high-pass 6 Hz': highpass(x, 6),
    'low-pass 90 Hz': lowpass(x, 90),
    'band-pass 6-90 Hz': bandpass(x, 6, 90),
    'band-pass + notch 50': notch(bandpass(x, 6, 90), 50),
    'band-pass + notch 60': notch(bandpass(x, 6, 90), 60),
}

print(f"{len(versoes)} versões prontas em {time.time() - inicio:.2f} s; cada uma com shape {x.shape} (canais, amostras, blocos, condições)")
for nome in versoes:
    print(" -", nome)

6 versões prontas em 0.26 s; cada uma com shape (9, 750, 4, 40) (canais, amostras, blocos, condições)
 - original
 - high-pass 6 Hz
 - low-pass 90 Hz
 - band-pass 6-90 Hz
 - band-pass + notch 50
 - band-pass + notch 60


## 4. Antes × depois

Pra cada versão: Root Mean Square do sinal pra medir a energia média do sinal em µV, potência em algumas faixas (cortes com Welch) e Signal to Noise Ratio na frequência do estímulo pra comparar sinal que interessa com ruido ao redor

In [7]:
def potencia_faixa(x, f1, f2):
    f, p = signal.welch(x, fs=FS, nperseg=250, axis=1)     # resolução de 1 Hz
    sel = (f >= f1) & (f <= f2)
    return p[:, sel].mean()


faixas = [(3, 6), (8, 16), (16, 48), (48, 52), (58, 62), (90, 125)]
print(f"{'versão':22s} {'RMS µV':>7s}  " + '  '.join(f'{a}-{b} Hz'.rjust(9) for a, b in faixas))
for nome, v in versoes.items():
    rms = np.sqrt((v ** 2).mean())
    pots = [potencia_faixa(v, a, b) for a, b in faixas]
    print(f"{nome:22s} {rms:7.2f}  " + '  '.join(f'{p:9.4f}' for p in pots))

versão                  RMS µV     3-6 Hz    8-16 Hz   16-48 Hz   48-52 Hz   58-62 Hz  90-125 Hz


original                  8.37     1.2940     2.3969     1.0420     0.2761     0.3873     0.0144


high-pass 6 Hz            7.99     0.1372     2.3252     1.0419     0.2761     0.3873     0.0144


low-pass 90 Hz            8.30     1.2940     2.3969     1.0420     0.2760     0.3861     0.0012
band-pass 6-90 Hz         7.95     0.1362     2.3459     1.0420     0.2761     0.3870     0.0012


band-pass + notch 50      7.89     0.1362     2.3454     1.0302     0.1744     0.3810     0.0012
band-pass + notch 60      7.84     0.1362     2.3456     1.0401     0.2705     0.1402     0.0012


In [8]:
def snr_estimulo(x):
    # x: (canais, T, blocos, condições). Média dos canais, janela de 2 s, FFT com resolução de 0,25 Hz
    sinal = x[:, ONSET:ONSET + 2 * FS].mean(axis=0)              # (500, blocos, condições)
    espectro = np.abs(np.fft.rfft(sinal, n=1000, axis=0))
    f = np.fft.rfftfreq(1000, 1 / FS)
    snrs = []
    for c in range(n_cond):
        k = np.argmin(np.abs(f - freqs[c]))
        vizinhos = np.r_[espectro[k - 4:k - 1, :, c], espectro[k + 2:k + 5, :, c]].mean(axis=0)
        snrs.append(20 * np.log10(espectro[k, :, c] / vizinhos))
    return np.array(snrs)                                         # (condições, blocos)


for nome, v in versoes.items():
    s = snr_estimulo(v)
    print(f"{nome:22s} SNR médio = {s.mean():5.2f} dB | ensaios com SNR > 0: {(s > 0).sum()} de 160")

original               SNR médio =  7.47 dB | ensaios com SNR > 0: 154 de 160


high-pass 6 Hz         SNR médio =  7.54 dB | ensaios com SNR > 0: 155 de 160
low-pass 90 Hz         SNR médio =  7.47 dB | ensaios com SNR > 0: 154 de 160
band-pass 6-90 Hz      SNR médio =  7.54 dB | ensaios com SNR > 0: 155 de 160
band-pass + notch 50   SNR médio =  7.54 dB | ensaios com SNR > 0: 155 de 160
band-pass + notch 60   SNR médio =  7.54 dB | ensaios com SNR > 0: 155 de 160


Faixas foram escolhidas para responder perguntas específicas:

3–6 Hz    : o que o high-pass deveria remover                     
8–16 Hz   : onde estão os estímulos; filtro não deve mexer    
16–48 Hz  : onde estão os harmônicos (2f, 3f)                     
48–52 Hz  : região da rede de 50 Hz                               
58–62 Hz  : região da rede de 60 Hz                               
90–125 Hz : o que o low-pass deveria remover                      


## 5. Verificação do notch (50/60 Hz)

Compara a potência exatamente em 50 Hz e 60 Hz com a média dos vizinhos (±3 a 6 Hz). Valor muito negativo = já existe um entalhe; perto de 0 = nada ali; positivo = interferência.

In [9]:
f, p = signal.welch(eeg, fs=FS, nperseg=T, axis=1)
p = p.mean(axis=(2, 3))          # PSD médio por canal


def razao_db(p_canal, f0):
    centro = p_canal[np.argmin(np.abs(f - f0))]
    viz = p_canal[((f >= f0 - 6) & (f <= f0 - 3)) | ((f >= f0 + 3) & (f <= f0 + 6))].mean()
    return 10 * np.log10(centro / viz)


r50 = np.array([razao_db(p[i], 50) for i in range(n_ch)])
r60 = np.array([razao_db(p[i], 60) for i in range(n_ch)])
print(f"50 Hz: mediana dos 64 canais = {np.median(r50):6.1f} dB (min {r50.min():.1f}, max {r50.max():.1f})")
print(f"60 Hz: mediana dos 64 canais = {np.median(r60):6.1f} dB (min {r60.min():.1f}, max {r60.max():.1f})")

50 Hz: mediana dos 64 canais =  -24.2 dB (min -25.0, max -13.1)
60 Hz: mediana dos 64 canais =    0.4 dB (min -6.2, max 2.9)


## 6. Conclusões (S1)

**Os dados são usáveis?** Sim, nos canais occipitais.
- Não há NaN nem Inf, e nenhum ensaio passa de 100 µV nos 9 canais occipitais (0 de 160).
- Canais fora do padrão:
  - **CZ** (0,43 µV) é a referência do equipamento e não tem sinal. C1, C2 e CPZ (1,5–2 µV) são baixos por estarem colados na referência.
  - **M2** (0,51 µV) está quase sem sinal. **M1** (24 µV) é ruidoso.
  - Nenhum desses entra na análise de SSVEP.
- Nos 64 canais, 48 de 160 ensaios passam de 100 µV. Os artefatos vêm de M1, T7 e dos frontais (FP1, FPZ, FP2, provavelmente piscadas). Por isso não é preciso rejeitar ensaios se só os occipitais forem usados.

**Filtros (antes × depois):**
- **High-pass 6 Hz:** remove quase toda a potência de 3–6 Hz (1,29 → 0,14). Não mexe na faixa do estímulo (8–16 Hz) e sobe o SNR só de 7,47 para 7,54 dB.
- **Low-pass 90 Hz:** só atua acima de 90 Hz, onde quase não há energia. Não muda o SNR.
- **Band-pass 6–90 Hz:** junta os dois efeitos. É uma limpeza leve, opcional, porque o dado já vem filtrado em 3–100 Hz.

**Notch:**
- **50 Hz:** já existe um entalhe de cerca de −24 dB em todos os canais (*notch* de hardware). Aplicar outro não traz ganho.
- **60 Hz:** não há interferência (≈ 0 dB na mediana). O desvio mais forte vem de M1, que é ruidoso. Um *notch* em 60 Hz só removeria sinal limpo.
- **Decisão:** não aplicar *notch*.
